In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class ConvBNLeakyReLU(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1):
        super().__init__()
        padding = 1 if kernel_size == 3 else 0
        self.convbnleakyrelu=nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(negative_slope=0.1, inplace=True),
        )

    def forward(self, x):
        return self.convbnleakyrelu(x)

In [3]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.residual = nn.Sequential(
            ConvBNLeakyReLU(channels, channels // 2, 1, 1),
            ConvBNLeakyReLU(channels // 2, channels, 3, 1),
        )

    def forward(self, x):
        return x + self.residual(x)

In [ ]:
class YOLOv3(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_initial = ConvBNLeakyReLU(3, 32, 3, 1)
        self.down1 = ConvBNLeakyReLU(32, 64, 3, 2)
        self.stage1 = ResidualBlock(64)
        self.down2 = ConvBNLeakyReLU(64, 128, 3, 2)
        self.stage2 = nn.Sequential(
            ResidualBlock(128),
            ResidualBlock(128),
        )
        self.down3 = ConvBNLeakyReLU(128, 256, 3, 2)
        self.stage3 = nn.Sequential(
            ResidualBlock(256),
            ResidualBlock(256),
            ResidualBlock(256),
            ResidualBlock(256),
            ResidualBlock(256),
            ResidualBlock(256),
            ResidualBlock(256),
            ResidualBlock(256),
        )
        self.down4 = ConvBNLeakyReLU(256, 512, 3, 2)
        self.stage4 = nn.Sequential(
            ResidualBlock(512),
            ResidualBlock(512),
            ResidualBlock(512),
            ResidualBlock(512),
            ResidualBlock(512),
            ResidualBlock(512),
            ResidualBlock(512),
            ResidualBlock(512),
        )
        self.down5 = ConvBNLeakyReLU(512, 1024, 3, 2)
        self.stage5 = nn.Sequential(
            ResidualBlock(1024),
            ResidualBlock(1024),
            ResidualBlock(1024),
            ResidualBlock(1024),
        )
        self.head13 = nn.Sequential(
            ConvBNLeakyReLU(1024, 512, 1, 1),
            ConvBNLeakyReLU(512, 1024, 3, 1),
            ConvBNLeakyReLU(1024, 512, 1, 1),
            ConvBNLeakyReLU(512, 1024, 3, 1),
            ConvBNLeakyReLU(1024, 512, 1, 1),
        )
        self.head13_conv6 = ConvBNLeakyReLU(512, 1024, 3, 1)
        self.pred13 = nn.Conv2d(1024, 255, 1, 1)
        self.reduce13_to26 = ConvBNLeakyReLU(512, 256, 1, 1)
        self.upsample =nn.Upsample(scale_factor=2, mode='nearest')
        self.head26 = nn.Sequential(
            ConvBNLeakyReLU(768, 256, 1, 1),
            ConvBNLeakyReLU(256, 512, 3, 1),
            ConvBNLeakyReLU(512, 256, 1, 1),
            ConvBNLeakyReLU(256, 512, 3, 1),
            ConvBNLeakyReLU(512, 256, 1, 1),
        )
        self.head26_conv6 = ConvBNLeakyReLU(256, 512, 3, 1)
        self.pred26 = nn.Conv2d(512, 255, 1, 1)
        self.reduce26_to52= ConvBNLeakyReLU(256, 128, 1, 1)
        self.head52 = nn.Sequential(
            ConvBNLeakyReLU(384, 128, 1, 1),
            ConvBNLeakyReLU(128, 256, 3, 1),
            ConvBNLeakyReLU(256, 128, 1, 1),
            ConvBNLeakyReLU(128, 256, 3, 1),
            ConvBNLeakyReLU(256, 128, 1, 1),
        )
        self.head52_conv6 = ConvBNLeakyReLU(128, 256, 3, 1)
        self.pred52 = nn.Conv2d(256, 255, 1, 1)
    def forward(self, x):
        x = self.conv_initial(x)
        x = self.down1(x)
        x = self.stage1(x)
        x = self.down2(x)
        x = self.stage2(x)
        x = self.down3(x)
        x = self.stage3(x)
        route_52 = x.clone()
        x = self.down4(x)
        x = self.stage4(x)
        route_26 = x.clone()
        x = self.down5(x)
        x = self.stage5(x)
        x = self.head13(x)
        route_13 = x.clone()
        x = self.head13_conv6(x)
        x = self.pred13(x)
        prediction13 = x.clone()
        route_13 = self.upsample(self.reduce13_to26(route_13))
        x = torch.cat([route_13, route_26], 1)
        x = self.head26(x)
        head_route26 = x.clone()
        x = self.head26_conv6(x)
        x = self.pred26(x)
        prediction26 = x.clone()
        head_route26 = self.upsample(self.reduce26_to52(head_route26))
        x = torch.cat([route_52, head_route26], 1)
        x = self.head52(x)
        x = self.head52_conv6(x)
        x = self.pred52(x)
        prediction52 = x.clone()
        return prediction13, prediction26, prediction52